In [1]:
"""Independent verification of the test_structures.ipynb field reconstruction.

Cross-checks the tjf4x4-based field maps against an exact two-interface Airy
solution for the isotropic vacuum / C8H8 (200 nm) / Si stack at 250 eV.

Checks
------
1. s/p labeling of the 4x4 amplitudes (Airy comparison at a non-grazing angle
   where r_s and r_p differ strongly).
2. Phase/sign conventions of r and t, including the transmission phase
   reference plane.
3. Tangential-field continuity of the notebook reconstruction at both
   interfaces (s and p).
4. Back-interface consistency of the film forward/backward amplitudes with t.
5. A corrected p reconstruction (epsilon-weighted matching, 1/n^2 H->E map)
   for comparison.
"""

import sys

sys.path.insert(0, "/sessions/loving-compassionate-ritchie/mnt/refloxide/src")

import numpy as np
from periodictable.xsf import index_of_refraction

from refloxide.pxr.tjf4x4 import calculate_kz_uni, hc, uniaxial_reflectivity

ENERGY_EV = 250.0
FILM_A = 2000.0

n_vac = complex(1.0, 0.0)
n_film = index_of_refraction("C8H8", density=1.0, energy=ENERGY_EV * 1e-3)
n_si = index_of_refraction("Si", density=2.33, energy=ENERGY_EV * 1e-3)
wavelength_a = hc / ENERGY_EV
k0 = 2.0 * np.pi / wavelength_a

# Construct slabs/tensor exactly as stack_slabs/stack_tensor would
# (stacks.py needs py3.12; sandbox has 3.10 -- tjf4x4 itself imports fine).
def _slab_row(thick, n, rough):
    return [thick, 1.0 - n.real, n.imag, rough]


slabs = np.asarray(
    [_slab_row(0.0, n_vac, 0.0), _slab_row(FILM_A, n_film, 0.0), _slab_row(0.0, n_si, 0.0)],
    dtype=np.float64,
)
tensor = np.asarray(
    [np.diag([1.0 - n] * 3) for n in (n_vac, n_film, n_si)], dtype=np.complex128
)


def eps_lib(n):
    """Dielectric constant exactly as tjf4x4 builds it: conj(1 - 2*(1-n))."""
    return np.conj(1.0 - 2.0 * (1.0 - n))


def kx_from_q(q):
    return k0 * np.sqrt(1.0 - (q / (2.0 * k0)) ** 2 + 0j)


def kz_lib(n, kx):
    """Ordinary forward kz from the library branch (same as notebook helper)."""
    db = 1.0 - n
    row = np.conj(np.eye(3) - 2.0 * np.diag([db, db, db]))
    kz_all = calculate_kz_uni(
        row, np.asarray([kx], dtype=np.complex128), np.asarray([0.0]), k0
    )
    return complex(kz_all[0, 2])


def amplitudes_4x4(q):
    """(r_block0, t_block0, r_block1, t_block1) raw from the M matrix."""
    *_, m_full = uniaxial_reflectivity(
        np.asarray([q]), slabs, tensor, ENERGY_EV
    )
    m = m_full[0]
    denom = m[0, 0] * m[2, 2] - m[0, 2] * m[2, 0]
    r_b0 = (m[1, 0] * m[2, 2] - m[1, 2] * m[2, 0]) / denom  # notebook "r_ss"
    t_b0 = m[2, 2] / denom                                   # notebook "t_ss"
    r_b1 = (m[0, 0] * m[3, 2] - m[3, 0] * m[0, 2]) / denom  # notebook "r_pp"
    t_b1 = m[0, 0] / denom                                   # notebook "t_pp"
    return r_b0, t_b0, r_b1, t_b1


def airy(q, pol):
    """Exact 3-medium solution, e^{-iwt}, forward e^{+i kz z}.

    Returns r, t (back-interface phase reference), film F, B.
    s-pol amplitudes are E_y; p-pol amplitudes are H_y.
    """
    kx = kx_from_q(q)
    e0, e1, e2 = eps_lib(n_vac), eps_lib(n_film), eps_lib(n_si)
    kz = [np.sqrt(e * k0**2 - kx**2 + 0j) for e in (e0, e1, e2)]
    kz = [k if k.imag >= 0 else -k for k in kz]
    kz0, kz1, kz2 = kz
    if pol == "s":
        w0, w1, w2 = kz0, kz1, kz2
    else:  # p in H_y form: continuity of H_y and kz/eps * H_y
        w0, w1, w2 = kz0 / e0, kz1 / e1, kz2 / e2
    r01 = (w0 - w1) / (w0 + w1)
    t01 = 2.0 * w0 / (w0 + w1)
    r12 = (w1 - w2) / (w1 + w2)
    t12 = 2.0 * w1 / (w1 + w2)
    ph = np.exp(1j * kz1 * FILM_A)
    den = 1.0 + r01 * r12 * ph**2
    r = (r01 + r12 * ph**2) / den
    t = t01 * t12 * ph / den
    F = t01 / den
    B = r12 * ph**2 * F
    return r, t, F, B, (kz0, kz1, kz2)


# Notebook reconstruction helpers (verbatim logic) ---------------------------

def film_fb_notebook(r, kz0, kz1):
    fwd = 0.5 * ((1.0 + r) + (kz0 / kz1) * (1.0 - r))
    bwd = 0.5 * ((1.0 + r) - (kz0 / kz1) * (1.0 - r))
    return fwd, bwd


def s_field_notebook(q, z, r, t, kz0, kz1, kz2):
    fwd, bwd = film_fb_notebook(r, kz0, kz1)
    if z < 0.0:
        return np.exp(1j * kz0 * z) + r * np.exp(-1j * kz0 * z)
    if z <= FILM_A:
        return fwd * np.exp(1j * kz1 * z) + bwd * np.exp(-1j * kz1 * z)
    return t * np.exp(1j * kz2 * (z - FILM_A))


def p_exez_notebook(hy, kx, kz, n):
    scale = kz / (n * k0**2)
    return hy * scale, -hy * kx / (n * k0**2)


def p_field_notebook(q, z, r, t, kz0, kz1, kz2, kx):
    fwd, bwd = film_fb_notebook(r, kz0, kz1)
    if z < 0.0:
        hy = k0 * (np.exp(1j * kz0 * z) + r * np.exp(-1j * kz0 * z))
        return hy, *p_exez_notebook(hy, kx, kz0, n_vac)
    if z <= FILM_A:
        hy = k0 * (fwd * np.exp(1j * kz1 * z) + bwd * np.exp(-1j * kz1 * z))
        return hy, *p_exez_notebook(hy, kx, kz1, n_film)
    hy = k0 * t * np.exp(1j * kz2 * (z - FILM_A))
    return hy, *p_exez_notebook(hy, kx, kz2, n_si)


# ---------------------------------------------------------------------------
q_c = 2.0 * k0 * np.sqrt(2.0 * (1.0 - n_film.real))
q_test = [("q_c", q_c), ("q_mid", 0.05), ("q_large", 0.15)]

print("=" * 78)
print("CHECK 1+2: 4x4 amplitudes vs exact Airy (label, sign, phase reference)")
print("=" * 78)
for tag, q in q_test:
    th_g = np.degrees(np.arcsin(q / (2 * k0)))
    r_b0, t_b0, r_b1, t_b1 = amplitudes_4x4(q)
    rs, ts, *_ = airy(q, "s")
    rp, tp, *_ = airy(q, "p")
    print(f"\n{tag}: q={q:.5f} A^-1, grazing angle = {th_g:.2f} deg")
    print(f"  |r_s|^2 Airy = {abs(rs)**2:.6f}   |r_p|^2 Airy = {abs(rp)**2:.6f}")
    print(f"  block0 ('ss' in notebook): |r|^2 = {abs(r_b0)**2:.6f}")
    print(f"  block1 ('pp' in notebook): |r|^2 = {abs(r_b1)**2:.6f}")
    for name, rv, tv in [("block0", r_b0, t_b0), ("block1", r_b1, t_b1)]:
        for pol, ra, ta in [("s", rs, ts), ("p", rp, tp)]:
            dr = min(abs(rv - ra), abs(rv + ra))
            sgn = "+" if abs(rv - ra) < abs(rv + ra) else "-"
            print(f"    {name} vs {pol}-Airy: |r| mismatch {abs(abs(rv)-abs(ra)):.2e}, "
                  f"complex (best sign {sgn}) {dr:.2e}; "
                  f"t mismatch {min(abs(tv-ta), abs(tv+ta)):.2e}")

print()
print("=" * 78)
print("CHECK 3+4: notebook reconstruction continuity (using its own r,t labels)")
print("=" * 78)
eps_d = 1e-9  # A
for tag, q in q_test:
    kx = kx_from_q(q)
    kz0, kz1, kz2 = kz_lib(n_vac, kx), kz_lib(n_film, kx), kz_lib(n_si, kx)
    r_b0, t_b0, r_b1, t_b1 = amplitudes_4x4(q)

    # s reconstruction as the notebook does it (block0 amplitudes)
    e_top_v = s_field_notebook(q, -eps_d, r_b0, t_b0, kz0, kz1, kz2)
    e_top_f = s_field_notebook(q, +eps_d, r_b0, t_b0, kz0, kz1, kz2)
    e_bot_f = s_field_notebook(q, FILM_A - eps_d, r_b0, t_b0, kz0, kz1, kz2)
    e_bot_s = s_field_notebook(q, FILM_A + eps_d, r_b0, t_b0, kz0, kz1, kz2)
    print(f"\n{tag}: s-pol E_y jump  z=0: {abs(e_top_v-e_top_f)/abs(e_top_v):.2e}   "
          f"z=d: {abs(e_bot_f-e_bot_s)/max(abs(e_bot_f),1e-30):.2e}")

    # p reconstruction as the notebook does it (block1 amplitudes)
    h_tv, ex_tv, _ = p_field_notebook(q, -eps_d, r_b1, t_b1, kz0, kz1, kz2, kx)
    h_tf, ex_tf, _ = p_field_notebook(q, +eps_d, r_b1, t_b1, kz0, kz1, kz2, kx)
    h_bf, ex_bf, _ = p_field_notebook(q, FILM_A - eps_d, r_b1, t_b1, kz0, kz1, kz2, kx)
    h_bs, ex_bs, _ = p_field_notebook(q, FILM_A + eps_d, r_b1, t_b1, kz0, kz1, kz2, kx)
    print(f"        p-pol H_y jump  z=0: {abs(h_tv-h_tf)/abs(h_tv):.2e}   "
          f"z=d: {abs(h_bf-h_bs)/max(abs(h_bf),1e-30):.2e}")
    print(f"        p-pol E_x jump  z=0: {abs(ex_tv-ex_tf)/max(abs(ex_tv),1e-30):.2e}   "
          f"z=d: {abs(ex_bf-ex_bs)/max(abs(ex_bf),1e-30):.2e}")

print()
print("=" * 78)
print("CHECK 5: corrected p matching (eps-weighted) + exact Airy film amplitudes")
print("=" * 78)
for tag, q in q_test:
    kx = kx_from_q(q)
    rs, ts, Fs, Bs, (kz0, kz1, kz2) = airy(q, "s")
    rp, tp, Fp, Bp, _ = airy(q, "p")
    # notebook F,B for s vs exact Airy F,B
    fwd_s, bwd_s = film_fb_notebook(rs, kz0, kz1)
    # eps-weighted matching for p (H_y): F-B uses (kz0/e0)/(kz1/e1)
    w_ratio = (kz0 / eps_lib(n_vac)) / (kz1 / eps_lib(n_film))
    fwd_p = 0.5 * ((1.0 + rp) + w_ratio * (1.0 - rp))
    bwd_p = 0.5 * ((1.0 + rp) - w_ratio * (1.0 - rp))
    print(f"\n{tag}:")
    print(f"  s: notebook F,B vs Airy F,B err = {abs(fwd_s-Fs):.2e}, {abs(bwd_s-Bs):.2e}")
    print(f"  p: kz-only F,B vs Airy err     = "
          f"{abs(film_fb_notebook(rp, kz0, kz1)[0]-Fp):.2e}, "
          f"{abs(film_fb_notebook(rp, kz0, kz1)[1]-Bp):.2e}")
    print(f"  p: eps-weighted F,B vs Airy err = {abs(fwd_p-Fp):.2e}, {abs(bwd_p-Bp):.2e}")
    # back-interface consistency
    ph = np.exp(1j * kz1 * FILM_A)
    print(f"  s: F*ph + B/ph vs t err = {abs(Fs*ph + Bs/ph - ts):.2e}")

print()
print("=" * 78)
print("CHECK 6: angle bookkeeping")
print("=" * 78)
for tag, q in [("q_c", q_c)]:
    th = np.degrees(np.arcsin(np.clip(q / (2 * k0), -1, 1)))
    print(f"q={q:.5f}: arcsin(q/2k0) = {th:.3f} deg  -> this is the GRAZING angle "
          f"(from the surface); from the normal it is {90-th:.3f} deg")
    kx = kx_from_q(q)
    kz0 = kz_lib(n_vac, kx)
    print(f"check: kx/k0 = {abs(kx)/k0:.5f} = cos({th:.3f} deg) = "
          f"{np.cos(np.radians(th)):.5f}; kz/k0 = {abs(kz0)/k0:.5f} = "
          f"sin({th:.3f} deg) = {np.sin(np.radians(th)):.5f}")

CHECK 1+2: 4x4 amplitudes vs exact Airy (label, sign, phase reference)

q_c: q=0.01624 A^-1, grazing angle = 3.67 deg
  |r_s|^2 Airy = 0.532659   |r_p|^2 Airy = 0.531252
  block0 ('ss' in notebook): |r|^2 = 0.531252
  block1 ('pp' in notebook): |r|^2 = 0.532659
    block0 vs s-Airy: |r| mismatch 9.65e-04, complex (best sign +) 1.35e-03; t mismatch 3.20e-04
    block0 vs p-Airy: |r| mismatch 0.00e+00, complex (best sign +) 1.67e-16; t mismatch 4.88e-04
    block1 vs s-Airy: |r| mismatch 1.11e-16, complex (best sign +) 2.50e-16; t mismatch 1.34e-04
    block1 vs p-Airy: |r| mismatch 9.65e-04, complex (best sign +) 1.35e-03; t mismatch 2.58e-04

q_mid: q=0.05000 A^-1, grazing angle = 11.38 deg
  |r_s|^2 Airy = 0.008731   |r_p|^2 Airy = 0.007608
  block0 ('ss' in notebook): |r|^2 = 0.007608
  block1 ('pp' in notebook): |r|^2 = 0.008731
    block0 vs s-Airy: |r| mismatch 6.22e-03, complex (best sign +) 6.27e-03; t mismatch 3.52e-03
    block0 vs p-Airy: |r| mismatch 5.55e-17, complex (best 